# Base 1 — Gráficos exploratórios e STL

A exploração usa os dados preparados. A STL é aplicada somente ao trecho de treino, sem consultar o teste.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from series_temporais.reporting.graficos_exploratorios import aplicar_estilo, plotar_acf

aplicar_estilo()

def seasonal_strength(result):
    denom = np.nanvar(result.seasonal + result.resid)
    return np.nan if denom == 0 else max(0.0, 1 - np.nanvar(result.resid) / denom)

def trend_strength(result):
    denom = np.nanvar(result.trend + result.resid)
    return np.nan if denom == 0 else max(0.0, 1 - np.nanvar(result.resid) / denom)

BASE_DIR = ROOT / 'trabalho/bases/grupo1'
df = pd.read_csv(BASE_DIR / 'base1_limpa_preparada.csv', parse_dates=['Date'])
train = pd.read_csv(BASE_DIR / 'base1_treino_preparada.csv', parse_dates=['Date'])
df['retorno_pct'] = df['Close'].pct_change() * 100
display(df[['Close', 'Volume', 'retorno_pct']].describe().T)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=False)
axes[0].plot(df['Date'], df['Close'], lw=.8, color='#e69f00')
axes[0].set(title='Bitcoin — preço de fechamento diário', ylabel='USD/BTC')
recent = df.tail(180)
axes[1].plot(recent['Date'], recent['Close'], lw=1, color='#0072b2')
axes[1].set(title='Bitcoin — últimos 180 dias da base', ylabel='USD/BTC', xlabel='Data')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(df['retorno_pct'].dropna(), bins=60, ax=axes[0], color='#009e73')
axes[0].set(title='Distribuição dos retornos diários', xlabel='Retorno (%)')
axes[1].plot(df['Date'], np.log1p(df['Volume']), lw=.7, color='#cc79a7')
axes[1].set(title='Volume diário em escala logarítmica', xlabel='Data', ylabel='log(1 + volume)')
plt.tight_layout(); plt.show()

In [ ]:
weekday = df.assign(dia=df['Date'].dt.day_name()).groupby('dia')['retorno_pct'].mean().reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
weekday.plot.bar(ax=axes[0], color='#56b4e9')
axes[0].set(title='Retorno médio por dia da semana', xlabel='', ylabel='Retorno médio (%)')
plotar_acf(axes[1], df['retorno_pct'].dropna().to_numpy(), lags=40, unidade='dias')
axes[1].set_title('ACF dos retornos diários')
plt.tight_layout(); plt.show()

In [ ]:
serie = train.set_index('Date')['Close'].asfreq('D')
results = {}
for label, period in {'semanal (7)': 7, 'anual (365)': 365}.items():
    result = STL(serie, period=period, robust=True).fit()
    results[label] = result
    result.plot().set_size_inches(15, 8)
    plt.suptitle(f'Base 1 — STL {label}', y=1.02)
    plt.tight_layout(); plt.show()
display(pd.DataFrame([{'período': k, 'observações/ciclo': 7 if '7' in k else 365, 'força sazonal': seasonal_strength(v), 'força tendência': trend_strength(v)} for k, v in results.items()]).round(4))

## Interpretação

A força sazonal é calculada por `max(0, 1 - Var(resíduo) / Var(sazonal + resíduo))`. Os períodos de 7 e 365 dias são candidatos exploratórios; sua escolha para um modelo depende também da validação walk-forward.